In [1]:
import pandas as pd
import numpy as np

TARGETS = ["Y"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3", 
    "LSG_1",
    "LSG_2"  
]

results = pd.read_excel("resultados.xlsx")


In [2]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [3]:
df_summary_top10

,dataset,target,mean_r2,std_r2,mean_mse,std_mse
0,Train_1,Y,0.993951,0.000197,0.000505,0.000016
1,Train_2,Y,0.995144,0.000424,0.000518,0.000045
2,Val,Y,0.987339,0.003103,0.000940,0.000230
3,Test_1,Y,0.987850,0.002636,0.000991,0.000215
4,Test_2,Y,0.988814,0.000933,0.001177,0.000098
5,Test_3,Y,0.892815,0.048810,0.008204,0.003736


In [4]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
print(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====
  Target  Dataset                          Best_Model        Neurons   Best_R2
0      Y  Train_1         model_arch16-8_r0.9_seed421        [16, 8]  0.994267
1      Y  Train_2      model_arch64-32_r0.01_seed3631       [64, 32]  0.995738
2      Y      Val      model_arch64-32_r0.01_seed3631       [64, 32]  0.992585
3      Y   Test_1      model_arch16-8-4_r0.9_seed7576     [16, 8, 4]  0.992642
4      Y   Test_2          model_arch16_r0.01_seed421           [16]  0.990529
5      Y   Test_3  model_arch128-64-32_r0.01_seed7576  [128, 64, 32]  0.967709
